In [13]:
import os
import json
from collections import defaultdict
from pathlib import Path


SAVE = False  

# === 1. Folder path ===
folder_path = "./output"

# === 2. Containers ===
pattern_to_labels = defaultdict(set)            # pattern -> set(labels)
pattern_label_sources = defaultdict(lambda: defaultdict(set))  # pattern -> label -> files

# === 3. Collect from JSON files ===
for file_name in os.listdir(folder_path):
    if not file_name.endswith(".json"):
        continue
    file_path = os.path.join(folder_path, file_name)
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        print(f"[Skip] Invalid JSON: {file_name}")
        continue

    if isinstance(data, list):
        for doc in data:
            for entity in doc.get("entities", []):
                pattern = entity.get("pattern")
                label = entity.get("label")
                if not pattern or not label:
                    continue
                pattern_to_labels[pattern].add(label)
                pattern_label_sources[pattern][label].add(file_name)

# === 4. Report initial collisions (before priority) ===
multi_label_patterns = {p: labels for p, labels in pattern_to_labels.items() if len(labels) > 1}
print("=" * 80)
print(f"Initial patterns with multiple labels: {len(multi_label_patterns)}")
print("=" * 80)
if num_collisions == 0:
    print("No multi-label patterns found.")
else:
    for p in sorted(multi_label_patterns.keys(), key=str.lower):
        labels = sorted(list(multi_label_patterns[p]), key=str.lower)
        print(f"\n Pattern: {p}")
        print(f"   Labels ({len(labels)}): {', '.join(labels)}")
        for lbl in labels:
            sources = sorted(list(pattern_label_sources[p][lbl]))
            if sources:
                print(f"   - {lbl}  ←  from files: {', '.join(sources)}")
            else:
                print(f"   - {lbl}")

# === 5. Priority table + keyword overrides ===
label_priority = [
    # Domain / Sector
    "AGRICULTURE","EDUCATION","HEALTH","ENERGY","ENVIRONMENT",
    "TECHNOLOGY","TRANSPORTATION","SECURITY","DEFENSE","ECONOMY",
    # Institution
    "ORGANIZATION","GOVERNMENT","AGENCY","COMPANY",
    # Policy / Program
    "POLICY","PROGRAM","STRATEGY","INITIATIVE",
    # Event / Document
    "EVENT","PROJECT","PLAN","REPORT",
    # Misc / Object
    "STANDARD","PRODUCT","ROLE","METRIC","TOOL"
]
priority_rank = {lbl: i for i, lbl in enumerate(label_priority)}

def choose_label(labels, pattern_text):
    low = (pattern_text or "").lower()
    if "policy" in low and "POLICY" in labels: return "POLICY"
    if "program" in low and "PROGRAM" in labels: return "PROGRAM"
    if "standard" in low and "STANDARD" in labels: return "STANDARD"
    if "report"  in low and "REPORT"  in labels: return "REPORT"
    def rank(lbl): return priority_rank.get(lbl, len(label_priority)+999)
    return sorted(labels, key=rank)[0]

# === 6. Apply priority  ===
chosen_map = {}   # pattern -> chosen_label
dropped = []      # for review

for pattern, labels in pattern_to_labels.items():
    chosen = choose_label(labels, pattern)
    chosen_map[pattern] = chosen
    for lbl in labels:
        if lbl != chosen:
            dropped.append({
                "pattern": pattern,
                "kept": chosen,
                "dropped_label": lbl,
                "sources": sorted(list(pattern_label_sources[pattern][lbl]))
            })


# === 8. If everything looks good, you can SAVE by flipping SAVE=True ===
if SAVE:
    out_dir = Path(folder_path).parent / "universal_pattern_output"
    out_dir.mkdir(parents=True, exist_ok=True)

    final_json   = out_dir / "unique_entities_final.json"
    conflict_json= out_dir / "multi_label_conflicts.json"
    dropped_json = out_dir / "dropped_labels.json"

    unique_entities_final = [{"pattern": p, "label": lbl} for p, lbl in sorted(chosen_map.items(), key=lambda x: x[0].lower())]
    conflict_payload = {p: sorted(list(labs)) for p, labs in sorted(multi_label_patterns.items(), key=lambda x: x[0].lower())}

    with open(final_json, "w", encoding="utf-8") as f:
        json.dump(unique_entities_final, f, indent=2, ensure_ascii=False)
    with open(conflict_json, "w", encoding="utf-8") as f:
        json.dump(conflict_payload, f, indent=2, ensure_ascii=False)
    with open(dropped_json, "w", encoding="utf-8") as f:
        json.dump(dropped, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 80)
    print(f" Saved:")
    print(f"  - Final:    {final_json}")
    print(f"  - Conflicts:{conflict_json}")
    print(f"  - Dropped:  {dropped_json}")
else:
    print("\n Files NOT saved. Review the checks above. Set SAVE=True to write outputs.")


Initial patterns with multiple labels: 211

 Pattern: accuracy
   Labels (2): BENCHMARK, STANDARD
   - BENCHMARK  ←  from files: 381-390.json
   - STANDARD  ←  from files: 231-240.json

 Pattern: advanced air mobility
   Labels (2): TECHNOLOGY, TRANSPORTATION
   - TECHNOLOGY  ←  from files: 631-640.json
   - TRANSPORTATION  ←  from files: 561-570.json, 571-580.json

 Pattern: advisory council
   Labels (2): ORGANIZATION, PROGRAM
   - ORGANIZATION  ←  from files: 451-460.json
   - PROGRAM  ←  from files: 281-290.json

 Pattern: agency use of AI
   Labels (2): GOVERNMENT, POLICY
   - GOVERNMENT  ←  from files: 271-280.json
   - POLICY  ←  from files: 741-750.json

 Pattern: AGI
   Labels (2): FRONTIER_AI, GENERAL_PURPOSE_AI
   - FRONTIER_AI  ←  from files: 451-460.json
   - GENERAL_PURPOSE_AI  ←  from files: 481-490.json

 Pattern: agriculture
   Labels (2): AGRICULTURE, PROGRAM
   - AGRICULTURE  ←  from files: 681-690.json
   - PROGRAM  ←  from files: 761-770.json

 Pattern: AI adoption

In [12]:
dropped

[{'pattern': 'specialty crop production',
  'kept': 'AGRICULTURE',
  'dropped_label': 'PROGRAM',
  'sources': ['771-780.json']},
 {'pattern': 'metadata',
  'kept': 'TECHNOLOGY',
  'dropped_label': 'SENSITIVE_DATA',
  'sources': ['831-840.json']},
 {'pattern': 'metadata',
  'kept': 'TECHNOLOGY',
  'dropped_label': 'DATA',
  'sources': ['771-780.json']},
 {'pattern': 'content moderation',
  'kept': 'POLICY',
  'dropped_label': 'MECHANISM',
  'sources': ['441-450.json', '581-590.json', '601-610.json']},
 {'pattern': 'data management',
  'kept': 'TECHNOLOGY',
  'dropped_label': 'ACCOUNTABILITY',
  'sources': ['451-460.json']},
 {'pattern': 'AI professionals',
  'kept': 'STAKEHOLDER',
  'dropped_label': 'EMPLOYMENT',
  'sources': ['271-280.json']},
 {'pattern': 'Armed Forces',
  'kept': 'ORGANIZATION',
  'dropped_label': 'MILITARY',
  'sources': ['021-030.json', '361-370.json', '781-790.json']},
 {'pattern': 'policy recommendations',
  'kept': 'POLICY',
  'dropped_label': 'DOCUMENT',
  'sou

In [ ]:
problems = []
for pattern, labels in pattern_to_labels.items():
    chosen = chosen_map[pattern]
    if chosen not in labels:
        problems.append((pattern, chosen, sorted(list(labels))))

residual_conflicts = {p: labs for p, labs in pattern_to_labels.items() if len(labs) > 1 and chosen_map[p] not in labs}

print("Post-priority checks:")
print(f"- Total unique patterns: {len(chosen_map)}")
print(f"- Dropped label records: {len(dropped)}")
print(f"- Inconsistencies (chosen not in original labels): {len(problems)}")
print(f"- Residual multi-label patterns (should be 0): {len(residual_conflicts)}")

if problems:
    print("\n Inconsistencies examples:")
    for pattern, chosen, labs in problems[:10]:
        print(f"  Pattern: {pattern} | chosen={chosen} | original_labels={labs}")

if residual_conflicts:
    print("\n Residual multi-label patterns examples:")
    for p, labs in list(residual_conflicts.items())[:10]:
        print(f"  {p}: {sorted(list(labs))} | chosen={chosen_map[p]}")


Post-priority checks (no files saved yet):
- Total unique patterns: 4448
- Dropped label records: 241
- Inconsistencies (chosen not in original labels): 0
- Residual multi-label patterns (should be 0): 0


In [3]:
import pandas as pd
entity_label = pd.read_parquet("./output/entity_label.parquet")
entity_label

,doc_index,AGORA ID,label,pattern
0,001,2425,ORGANIZATION,Department of Commerce
1,001,2425,ROLE,Secretary of Commerce
2,001,2425,TECHNOLOGY,AI
3,001,2425,TECHNOLOGY,automation technologies
4,001,2425,SAFETY,cybersecurity
...,...,...,...,...
8499,947,1196,MECHANISM,rapid and safe shut-down procedures
8500,947,1196,ALIGNMENT_RISK,AI alignment
8501,947,1196,MISUSE_RISK,malicious actors
8502,947,1196,RESEARCH_STAGE,AI safety research
